# 05 — 可视化报告

> 🎯 **适用场景**: 将分析结果转化为可直接放入报告的精美图表
> ⏱️ **预计学习时长**: 45 分钟
> 📌 **本章包含 8 个报告级图表模板**，每个模板 = 代码 + 配色 + 解读话术 + 🚫避坑

---

## 🗺️ 图表选择决策树

```
分析目的是什么？
├── 📊 比较 → 柱状图 / 条形图        → 模板 #2, #6
├── 📈 趋势 → 折线图 / 面积图        → 模板 #1
├── 🔗 占比 → 饼图 / 环形图 / 树图   → 模板 #2, #7
├── 📉 分布 → 直方图 / 箱线图        → 模板 #7
├── 🔗 关系 → 散点图 / 热力图        → 模板 #3, #5
└── 🎯 漏斗 → 漏斗图 / 桑基图        → 模板 #4
```

---

## 配色规范



In [ ]:
# ═══════════════════════════════════════════
# 环境准备
# ═══════════════════════════════════════════

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 150
sns.set_style('whitegrid')

# ── 统一配色方案 ──
BLUE   = '#1f77b4'
ORANGE = '#ff7f0e'
GREEN  = '#2ca02c'
RED    = '#d62728'
PURPLE = '#9467bd'
BROWN  = '#8c564b'

ECOMMERCE_COLORS = [BLUE, ORANGE, GREEN, RED, PURPLE, BROWN]
FUNNEL_COLORS    = ['#4CAF50', '#FF9800', '#FF5722', '#F44336']

DATA_DIR = "../data"
df_orders  = pd.read_csv(f"{DATA_DIR}/olist_orders_dataset.csv",
    parse_dates=['order_purchase_timestamp', 'order_delivered_customer_date',
                 'order_estimated_delivery_date'])
df_items   = pd.read_csv(f"{DATA_DIR}/olist_order_items_dataset.csv")
df_payments= pd.read_csv(f"{DATA_DIR}/olist_order_payments_dataset.csv")
df_products= pd.read_csv(f"{DATA_DIR}/olist_products_dataset.csv")

df_oi = df_orders.merge(df_items, on='order_id', how='inner')
order_pay = df_payments.groupby('order_id')['payment_value'].sum().reset_index()
df_oi = df_oi.merge(order_pay, on='order_id', how='left')
df_delivered = df_oi[df_oi['order_status']=='delivered'].copy()

print("✅ 数据准备完成")


## 模板 #1: 月度 GMV 趋势折线图（带移动平均）

> 🎨 **配色**: 蓝色折线 + 橙色移动平均 + 浅蓝填充
> 🚫 **避坑**: 折线图 y 轴不需要从 0 开始（柱状图才需要）；避免 3D 效果

In [ ]:
# 📌 模板 #1: 月度 GMV 趋势
df_delivered['year_month'] = df_delivered['order_purchase_timestamp'].dt.to_period('M')
monthly = df_delivered.groupby('year_month').agg(
    gmv=('price', 'sum'),
    orders=('order_id', 'nunique')
).reset_index()
monthly['year_month_str'] = monthly['year_month'].astype(str)
monthly['gmv_ma3'] = monthly['gmv'].rolling(3).mean()  # 3月移动平均

fig, ax = plt.subplots(figsize=(16, 7))

# 面积填充
ax.fill_between(range(len(monthly)), monthly['gmv']/1e6,
                alpha=0.12, color=BLUE)
# 原始折线
ax.plot(range(len(monthly)), monthly['gmv']/1e6,
        color=BLUE, linewidth=1.5, marker='o', markersize=4, label='月度GMV')
# 移动平均
ax.plot(range(len(monthly)), monthly['gmv_ma3']/1e6,
        color=ORANGE, linewidth=2.5, label='3月移动平均')

ax.set_title('月度 GMV 趋势（2016.09 - 2018.10）', fontsize=16, fontweight='bold', pad=15)
ax.set_ylabel('GMV (百万 R$)', fontsize=12)
ax.set_xticks(range(0, len(monthly), 2))
ax.set_xticklabels(monthly['year_month_str'].iloc[::2], rotation=45, ha='right')
ax.legend(fontsize=11, loc='upper left')
ax.grid(True, alpha=0.3)

# 标注峰值
max_idx = monthly['gmv'].idxmax()
ax.annotate(f"峰值: R$ {monthly['gmv'].iloc[max_idx]/1e6:.1f}M",
            xy=(max_idx, monthly['gmv'].iloc[max_idx]/1e6),
            xytext=(10, 20), textcoords='offset points', fontsize=10,
            arrowprops=dict(arrowstyle='->', color=RED),
            bbox=dict(boxstyle='round,pad=0.3', facecolor='yellow', alpha=0.3))

# 信息标注
ax.text(0.02, 0.95,
        f"总GMV: R$ {monthly['gmv'].sum()/1e6:.1f}M | "
        f"月均: R$ {monthly['gmv'].mean()/1e6:.1f}M | "
        f"月均订单: {monthly['orders'].mean():.0f}",
        transform=ax.transAxes, fontsize=10, va='top',
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

plt.tight_layout()
plt.savefig('../outputs/05_template01_gmv_trend.png', dpi=200, bbox_inches='tight')
plt.show()

# 📌 解读话术
print("📌 解读话术:")
print("  'Olist 的 GMV 从 2016 年底的约 R$ 10 万/月增长到 2018 年初的 R$ 100 万+/月'")
print("  '增长势头在 2017 年最强，2018 年趋于平稳'")
print("  '2018 年 8 月后数据可能不完整，需验证'")


## 模板 #2: 品类 GMV 占比（水平条形图）

> 🎨 **配色**: viridis 渐变色带
> 🚫 **避坑**: 类别 > 15 个时不要用饼图——改用条形图或树图

In [ ]:
# 📌 模板 #2: 品类 GMV Top 10 水平条形图
df_products = df_products.merge(
    pd.read_csv(f"{DATA_DIR}/product_category_name_translation.csv"),
    on='product_category_name', how='left')
df_products['cat_en'] = df_products['product_category_name_english'].fillna(
    df_products['product_category_name'])

df_oi_cat = df_oi.merge(df_products[['product_id','cat_en']], on='product_id', how='left')
cat_gmv = df_oi_cat.groupby('cat_en')['price'].sum().sort_values(ascending=False).head(10)

fig, ax = plt.subplots(figsize=(12, 7))
colors_cat = sns.color_palette("viridis", len(cat_gmv))

bars = ax.barh(range(len(cat_gmv)), cat_gmv.values[::-1]/1e6, color=colors_cat[::-1])
ax.set_yticks(range(len(cat_gmv)))
ax.set_yticklabels(cat_gmv.index[::-1])
ax.set_xlabel('GMV (百万 R$)')
ax.set_title('Top 10 品类 GMV 占比', fontsize=15, fontweight='bold')

for i, (v, pct) in enumerate(zip(cat_gmv.values[::-1], 
    cat_gmv.values[::-1]/cat_gmv.sum()*100)):
    ax.text(v/1e6 + 0.05, i, f'R$ {v/1e6:.1f}M ({pct:.1f}%)', va='center', fontsize=10)

# 信息标注
ax.text(0.98, 0.02, f"Top10 占总GMV: {cat_gmv.sum()/df_oi_cat['price'].sum()*100:.0f}%",
        transform=ax.transAxes, ha='right', fontsize=11,
        bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

plt.tight_layout()
plt.savefig('../outputs/05_template02_category_gmv.png', dpi=200, bbox_inches='tight')
plt.show()

print("📌 解读话术:")
print(f"  'Top 3 品类贡献了 {cat_gmv.head(3).sum()/cat_gmv.sum()*100:.0f}% 的 Top10 GMV'")
print(f"  '#1 {cat_gmv.index[0]} 是绝对主力，单独贡献 {cat_gmv.iloc[0]/cat_gmv.sum()*100:.0f}%'")


## 模板 #3: RFM 用户分层气泡图

> 🎨 **配色**: 4 色分层 + 气泡大小 = 消费金额
> 🚫 **避坑**: RFM 分层需要先定义阈值，可用分位数法或业务规则

In [ ]:
# 📌 模板 #3: RFM 分层气泡图
# R: 最近一次购买距今天数
# F: 购买频次
# M: 消费金额
snapshot = df_delivered['order_purchase_timestamp'].max() + pd.Timedelta(days=1)

rfm = df_delivered.groupby('customer_id').agg(
    recency=('order_purchase_timestamp', lambda x: (snapshot - x.max()).days),
    frequency=('order_id', 'nunique'),
    monetary=('price', 'sum')
).reset_index()

# 分层（基于分位数）
r_med, f_med = rfm['recency'].median(), rfm['frequency'].median()
conditions = [
    (rfm['recency'] <= r_med) & (rfm['frequency'] > f_med),
    (rfm['recency'] <= r_med) & (rfm['frequency'] <= f_med),
    (rfm['recency'] > r_med) & (rfm['frequency'] > f_med),
    (rfm['recency'] > r_med) & (rfm['frequency'] <= f_med),
]
labels = ['高价值', '活跃新客', '沉睡老客', '流失用户']
rfm['segment'] = np.select(conditions, labels, '一般')

segment_colors = {'高价值': GREEN, '活跃新客': BLUE, '沉睡老客': ORANGE, '流失用户': RED, '一般': '#999999'}

# 可视化
fig, ax = plt.subplots(figsize=(12, 8))
for seg, color in segment_colors.items():
    subset = rfm[rfm['segment'] == seg]
    ax.scatter(subset['frequency'], subset['monetary'],
               s=np.clip(subset['monetary']/10, 10, 500),
               c=color, label=seg, alpha=0.5, edgecolors='white', linewidth=0.5)

ax.set_xlabel('购买频次 (F)', fontsize=13)
ax.set_ylabel('消费金额 R$ (M)', fontsize=13)
ax.set_title('RFM 客户分层气泡图', fontsize=15, fontweight='bold')
ax.legend(title='客户分层', loc='lower right', fontsize=11)
ax.set_xlim(0, rfm['frequency'].quantile(0.99))
ax.set_ylim(0, rfm['monetary'].quantile(0.99))
ax.grid(True, alpha=0.3)

# 标注各分层占比
for seg, color in segment_colors.items():
    cnt = (rfm['segment'] == seg).sum()
    pct = cnt / len(rfm) * 100
    print(f"  {seg:6s}: {cnt:>8,} ({pct:5.1f}%)")

plt.tight_layout()
plt.savefig('../outputs/05_template03_rfm.png', dpi=200, bbox_inches='tight')
plt.show()

print("📌 解读话术:")
print("  '高价值用户仅占少数但贡献最大——这是典型的帕累托分布'")
print("  '沉睡老客（曾经活跃但现在沉默）是唤回营销的最佳目标人群'")


## 模板 #4: 转化漏斗图

> 🎨 **配色**: 绿 → 橙 → 红（从好到差）
> 🚫 **避坑**: 漏斗每层用不同的、有辨识度的颜色；不要用 3D 漏斗

In [ ]:
# 📌 模板 #4: 订单状态转化漏斗
status_mapping = {'created': '1.创建', 'approved': '2.审核',
                  'processing': '3.处理中', 'shipped': '4.已发货',
                  'delivered': '5.已交付'}
df_orders['stage'] = df_orders['order_status'].map(status_mapping)
funnel_data = df_orders['stage'].value_counts()[
    ['1.创建','2.审核','3.处理中','4.已发货','5.已交付']]

fig, ax = plt.subplots(figsize=(10, 8))

# 横向漏斗
stages = funnel_data.index.tolist()
values = funnel_data.values
colors_funnel = FUNNEL_COLORS + ['#2196F3']

# 计算柱宽度（最大为1）
bar_widths = values / values[0]

for i, (stage, val, w) in enumerate(zip(stages, values, bar_widths)):
    ax.barh(stage, w, height=0.6, color=colors_funnel[i], alpha=0.8)
    ax.text(w/2, i, f'{val:,}', ha='center', va='center', fontsize=13, fontweight='bold', color='white')
    # 转化率标注
    conv_rate = val / values[0] * 100
    ax.text(w + 0.02, i, f'{conv_rate:.1f}%', va='center', fontsize=11, color='black')

ax.set_xlim(0, 1.15)
ax.set_title('订单状态转化漏斗', fontsize=15, fontweight='bold')
ax.set_xlabel('')

# 箭头连接
for i in range(len(stages)-1):
    drop = (1 - values[i+1]/values[i]) * 100
    if drop > 0:
        ax.text(1.05, i + 0.5, f'↓ {drop:.1f}%', ha='center', fontsize=9, color=RED)

plt.tight_layout()
plt.savefig('../outputs/05_template04_funnel.png', dpi=200, bbox_inches='tight')
plt.show()

print("📌 解读话术:")
print("  '从创建到交付的转化率约 97%，平台履约能力良好'")
print("  '创建 → 审核环节几乎无损失——说明系统自动化程度较高'")


## 模板 #5: 配送时长 vs 评分气泡图

> 🎨 **配色**: 气泡大小 = 订单量，颜色 = 评分
> 🚫 **避坑**: 散点图数据量大时用 alpha 透明度 + 采样渲染

In [ ]:
# 📌 模板 #5: 配送时长 vs 评分
df_reviews = pd.read_csv(f"{DATA_DIR}/olist_order_reviews_dataset.csv")
df_orders_del = df_orders[df_orders['order_status']=='delivered'][
    ['order_id','order_purchase_timestamp','order_delivered_customer_date']]
df_rev_del = df_reviews.merge(df_orders_del, on='order_id')
df_rev_del['delivery_days'] = (df_rev_del['order_delivered_customer_date'] -
                                df_rev_del['order_purchase_timestamp']).dt.days
df_rev_del = df_rev_del[(df_rev_del['delivery_days']>=0) & (df_rev_del['delivery_days']<=60)]

# 按配送天数+评分分组
grouped = df_rev_del.groupby(['delivery_days','review_score']).size().reset_index(name='count')

fig, ax = plt.subplots(figsize=(14, 8))
scatter = ax.scatter(grouped['delivery_days'], grouped['review_score'],
                     s=np.clip(grouped['count']*2, 10, 500),
                     c=grouped['review_score'], cmap='RdYlGn',
                     alpha=0.6, edgecolors='white', linewidth=0.3)

# 趋势线
from numpy.polynomial.polynomial import polyfit
x_fit = np.arange(0, 61)
for score in range(1, 6):
    subset = df_rev_del[df_rev_del['review_score']==score]
    if len(subset) > 100:
        avg_by_day = subset.groupby('delivery_days').size()
        ax.plot(avg_by_day.index, [score]*len(avg_by_day), alpha=0.15,
                color='black', linewidth=avg_by_day.values/avg_by_day.values.max()*3)

cbar = plt.colorbar(scatter, ax=ax, label='评分')
ax.set_xlabel('配送时长 (天)', fontsize=13)
ax.set_ylabel('评分', fontsize=13)
ax.set_title('配送时长 vs 评分（气泡大小 = 订单量）', fontsize=15, fontweight='bold')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../outputs/05_template05_delivery_score.png', dpi=200, bbox_inches='tight')
plt.show()

print("📌 解读话术:")
print("  '配送在 7 天以内时，评分集中在 4-5 分'")
print("  '超过 15 天后，1-2 分的差评比例明显上升'")
print("  '物流速度是影响用户满意度的核心杠杆'")


## 模板 #6: 支付方式分布环形图

> 🎨 **配色**: 各色区分，环形图比饼图更现代
> 🚫 **避坑**: 类别 ≤ 5 时才用饼图/环形图；超过则用条形图

In [ ]:
# 📌 模板 #6: 支付方式环形图
pay_counts = df_payments['payment_type'].value_counts()
pay_colors = {'credit_card': BLUE, 'boleto': ORANGE, 'voucher': GREEN, 'debit_card': RED}

fig, ax = plt.subplots(figsize=(8, 8))
wedges, texts, autotexts = ax.pie(
    pay_counts.values, labels=None, autopct='%1.1f%%',
    startangle=90, pctdistance=0.82,
    colors=[pay_colors.get(k, '#999999') for k in pay_counts.index],
    textprops={'fontsize': 13, 'fontweight': 'bold'},
    wedgeprops={'linewidth': 2, 'edgecolor': 'white'})

# 中心空白 → 环形
centre_circle = plt.Circle((0, 0), 0.60, fc='white', linewidth=0)
ax.add_artist(centre_circle)

ax.text(0, 0, '支付
方式', ha='center', va='center', fontsize=14, fontweight='bold', color='#555')

ax.set_title('支付方式分布', fontsize=15, fontweight='bold')

# 自定义图例
legend_labels = [f'{k} ({v/1000:.0f}k)' for k, v in pay_counts.items()]
ax.legend(wedges, legend_labels, title='支付方式', loc='center left', bbox_to_anchor=(1, 0.5))
plt.tight_layout()
plt.savefig('../outputs/05_template06_payment_pie.png', dpi=200, bbox_inches='tight')
plt.show()

print("📌 解读话术:")
print("  '信用卡占主导（~74%），boleto（银行票据）次之（~14%）'")
print("  '借记卡 + voucher 合计不到 12%'")


## 模板 #7: 评分分布环形图

> 🎨 **配色**: 红→黄→绿（差评→好评）
> 🚫 **避坑**: 从 12 点方向开始排列（startangle=90）

In [ ]:
# 📌 模板 #7: 评分分布环形图
score_counts = df_reviews['review_score'].value_counts().sort_index()

fig, ax = plt.subplots(figsize=(8, 8))
score_colors = ['#d7191c','#fdae61','#fee08b','#a6d96a','#1a9641']

wedges, texts, autotexts = ax.pie(
    score_counts.values, labels=None, autopct='%1.1f%%',
    startangle=90, pctdistance=0.82,
    colors=score_colors,
    textprops={'fontsize': 12, 'fontweight': 'bold'},
    wedgeprops={'linewidth': 2, 'edgecolor': 'white'})

centre_circle = plt.Circle((0, 0), 0.58, fc='white')
ax.add_artist(centre_circle)

avg_score = df_reviews['review_score'].mean()
ax.text(0, 0, f'{avg_score:.1f}\n/ 5.0', ha='center', va='center',
        fontsize=20, fontweight='bold', color='#333')

ax.set_title('用户评分分布', fontsize=15, fontweight='bold')

legend_labels = [f'{s}分 ({c/1000:.0f}k)' for s, c in score_counts.items()]
ax.legend(wedges, legend_labels, title='评分', loc='center left', bbox_to_anchor=(1, 0.5))
plt.tight_layout()
plt.savefig('../outputs/05_template07_ratings_donut.png', dpi=200, bbox_inches='tight')
plt.show()


## 模板 #8: KPI 仪表盘概述图

> 🎨 **配色**: 各指标独立色块，适合做报告第一页

In [ ]:
# 📌 模板 #8: KPI 仪表盘
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Olist 电商 KPI 仪表盘 (2016.09 - 2018.10)', fontsize=18, fontweight='bold', y=1.01)

kpis = [
    ('总 GMV', f'R$ {total_gmv/1e6:.1f}M', f'{total_orders/1000:.0f}k 订单', GREEN),
    ('客单价 (AOV)', f'R$ {aov_overall:.0f}', '月波动 < 10%', BLUE),
    ('复购率', f'{repurchase_rate:.1f}%', f'{repurchase_cust/1000:.0f}k 复购用户', RED if repurchase_rate < 10 else GREEN),
    ('准时交付率', f'{on_time_rate:.1f}%', f'平均 {avg_days:.0f} 天', ORANGE if on_time_rate < 95 else GREEN),
    ('平均评分', f'{avg_score:.1f} / 5.0', f'{n_total/1000:.0f}k 评价', GREEN if avg_score >= 4.0 else ORANGE),
    ('MAU', f'{mau.max()/1000:.0f}k', f'峰值月: {mau.idxmax()}', PURPLE),
]

for i, (title, value, subtitle, color) in enumerate(kpis):
    ax = axes[i // 3, i % 3]
    ax.set_facecolor('#f8f9fa')

    # 大数字
    ax.text(0.5, 0.65, value, ha='center', va='center',
            fontsize=28, fontweight='bold', color=color, transform=ax.transAxes)
    # 标题
    ax.text(0.5, 0.85, title, ha='center', va='center',
            fontsize=14, fontweight='bold', color='#555', transform=ax.transAxes)
    # 副标题
    ax.text(0.5, 0.40, subtitle, ha='center', va='center',
            fontsize=11, color='#888', transform=ax.transAxes)
    # 底部横线
    ax.axhline(y=0.15, xmin=0.2, xmax=0.8, color=color, linewidth=3)
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axis('off')

plt.tight_layout()
plt.savefig('../outputs/05_template08_kpi_dashboard.png', dpi=200, bbox_inches='tight', facecolor='white')
plt.show()

print("📌 解读话术:")
print("  '这张仪表盘是一页纸报告的核心——领导看这6个数字就够了'")


## 🚫 可视化避坑总表

| 坑 | 为什么有问题 | 正确做法 |
|----|-------------|----------|
| 3D 饼图 | 视觉扭曲比例，不专业 | 环形图或条形图 |
| 饼图 > 6 类 | 难以区分小块 | 用条形图替代 |
| 柱状图 y 轴不从 0 开始 | 视觉上夸大差异 | y 轴必须从 0 起步 |
| 双轴图比例不一致 | 容易操控视觉观感 | 标注两个单位+明确颜色 |
| 颜色过多 (>8 种) | 阅读困难 | ≤ 6 种颜色 |
| 无标签/Legend | 读者看不懂 | 每个图表必须有标题+图例 |
| 红绿配色 | 8% 男性色盲 | 用蓝橙替代 |
| 导出分辨率低 | 放大后模糊 | ≥ 150 DPI |

---

## 📝 康奈尔笔记联动区

### 左侧栏（核心概念）
- **图表选择决策树**: 业务问题 → 图表类型
- **配色统一**: 一套配色贯穿整个报告
- **解读话术**: 每张图配套一句话解读

### 右侧栏（反思问题）
> 💭 为什么商业报告偏好蓝橙色系而非默认色？
> 💭 漏斗图中哪个环节的转化率提升空间最大？
> 💭 如果只有 3 张图给 CEO 看，你选哪 3 张？

### 底部栏（行动清单）
- [ ] 从 8 个模板中选 3-4 张放入最终报告
- [ ] 确认所有图表分辨率 ≥ 150 DPI
- [ ] 到 project01_reflections.md 总结可视化心得
